# Minimal Vision Transformer: Evaluation and Visualization (CIFAR-10)

This is the **evaluate** notebook paired with `vit_train.ipynb`. It loads
a checkpoint already trained by that notebook and only runs forward
passes (`torch.no_grad()`) -- no training happens here. Because training
and evaluation are split, adding a new diagnostic plot later only
requires editing `code/transformers/vit_evaluate.py` and re-running this
notebook against the existing checkpoint, in seconds, with no GPU
training step.

Visualizations produced below:
- test accuracy and a per-class confusion matrix
- training/test loss and accuracy curves (from the training log)
- sample test images with true vs. predicted class (`images [B,C,H,W]` -> `logits [B,10]`)
- the Transformer-specific one: last-layer **CLS-to-patch attention**,
  reshaped from `[N]` back into the `(H/P, W/P)` patch grid and overlaid
  on the input image -- shows which image patches the classification
  token actually attends to
- learned **position-embedding similarity**, showing whether training
  gave the positional embeddings 2D spatial structure

Core code lives in `code/transformers/vit_evaluate.py`; this notebook is
a thin orchestration layer.

## 1. Clone/pull repository

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code", "transformers"))

## 2. Mount Google Drive and point at the checkpoint/log produced by `vit_train.ipynb`

Paths below must match the `CHECKPOINT_PATH` / `LOG_PATH` / `DATA_ROOT`
used in `vit_train.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch

DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"
CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/vit_cifar10/checkpoint.pt"
LOG_PATH = "/content/drive/MyDrive/thesis_data/vit_cifar10/train_log.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/thesis_data/vit_cifar10/figures"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")

## 3. Load the trained model and the CIFAR-10 test set

In [ ]:
from vit_evaluate import load_model_and_config
from vit_train import make_cifar10_dataloaders, CIFAR10_CLASSES

model, config = load_model_and_config(CHECKPOINT_PATH, device)
_, test_loader = make_cifar10_dataloaders(DATA_ROOT, batch_size=256)
test_dataset = test_loader.dataset
print(f"config = {config}")
print(f"test set size = {len(test_dataset)}")

## 4. Test accuracy and confusion matrix

In [ ]:
import os
from vit_evaluate import compute_test_accuracy_and_confusion, plot_confusion_matrix

os.makedirs(OUTPUT_DIR, exist_ok=True)
accuracy, confusion = compute_test_accuracy_and_confusion(model, test_loader, device)
print(f"test accuracy: {accuracy:.4f}  (chance level = {1 / config['num_classes']:.4f})")

plot_confusion_matrix(confusion, CIFAR10_CLASSES, save_path=os.path.join(OUTPUT_DIR, "confusion_matrix.png"))

## 5. Training/test loss and accuracy curves

In [ ]:
from vit_evaluate import plot_training_curves

plot_training_curves(LOG_PATH, save_path=os.path.join(OUTPUT_DIR, "training_curves.png"))

## 6. Sample predictions

In [ ]:
from vit_evaluate import visualize_predictions

visualize_predictions(
    model, test_dataset, device, CIFAR10_CLASSES, num_examples=5,
    save_path=os.path.join(OUTPUT_DIR, "sample_predictions.png"),
)

## 7. CLS-to-patch attention maps

In [ ]:
from vit_evaluate import visualize_cls_attention

visualize_cls_attention(
    model, test_dataset, device, CIFAR10_CLASSES, num_examples=4,
    save_path=os.path.join(OUTPUT_DIR, "cls_attention.png"),
)

## 8. Position-embedding similarity (learned 2D structure)

Reuses `visualize_position_embedding_similarity` from
`vit_patch_embedding.py`: cosine similarity of a few reference patches'
learned position embeddings to every other patch, reshaped into the
patch grid. If training gave the model useful spatial structure, each
reference patch should be most similar to patches near it in the
grid.

In [ ]:
from vit_patch_embedding import visualize_position_embedding_similarity

mid_patch = (model.num_patches_per_side // 2) * model.num_patches_per_side + model.num_patches_per_side // 2
visualize_position_embedding_similarity(
    model.position_embedding.detach().cpu(), model.num_patches_per_side,
    reference_patch_indices=[0, mid_patch],
    save_path=os.path.join(OUTPUT_DIR, "position_embedding_similarity.png"),
)